# Extension Step 4 — GNN (DAGNN) Task Graph Classifier

Trains a `DAGClassifier` on task graph realizations using a fixed train/val/test
split at the recipe level (seed=42): **16 train / 4 val / 4 test recipes**.

The val set drives LR scheduling and early stopping.
The test set is evaluated once at the end on the best-val-AUC checkpoint.

**Prerequisites:**
- Step embeddings in `STEP_EMBEDDINGS_DIR` (output of Extension Step 1)
- Pre-fusion cache in `CACHE_DIR` (output of b3_hungarian_matching.ipynb, sim_threshold=0.30)
- Task graphs in `GRAPHS_DIR` (annotations submodule)
- EgoVLP checkpoint at `EGOVLP_CKPT`

**Output:**
- Best checkpoint in `STEP4_OUTPUT_DIR/checkpoints/best.pt`
- Split assignment in `STEP4_OUTPUT_DIR/split_info.json`
- Metrics in `STEP4_OUTPUT_DIR/results.csv` (val row + test row)

In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Path constants ────────────────────────────────────────────────────────
DRIVE_ROOT           = '/content/drive/MyDrive/AML_Project'
REPO_DIR             = '/content/code'
EGOVLP_REPO          = '/content/EgoVLP'

EGOVLP_CKPT          = f'{DRIVE_ROOT}/models/egovlp.pth'
STEP_EMBEDDINGS_DIR  = f'{DRIVE_ROOT}/step1/step_embeddings'
ANNOTATIONS_PATH     = f'{REPO_DIR}/annotations/annotation_json/complete_step_annotations.json'
GRAPHS_DIR           = f'{REPO_DIR}/annotations/task_graphs'

# Pre-fusion cache built by b3_hungarian_matching.ipynb (sim_threshold=0.30)
CACHE_DIR            = f'{DRIVE_ROOT}/step3/cache_thr030'
STEP4_OUTPUT_DIR     = f'{DRIVE_ROOT}/step4/results_thr030'

# local copy of step embeddings (faster I/O than Drive during training)
LOCAL_EMBEDDINGS_DIR = '/content/step_embeddings'

print(f'cache dir  : {CACHE_DIR}')
print(f'output dir : {STEP4_OUTPUT_DIR}')

In [ ]:
# ── 3. Clone repos (--recursive fetches annotations submodule) ────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}
!git clone https://github.com/showlab/EgoVLP.git {EGOVLP_REPO}

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
import torch, os
pt_version  = torch.__version__.split('+')[0]
cuda_str    = f"cu{torch.version.cuda.replace('.', '')}"
os.environ['TORCH'] = pt_version
os.environ['CUDA']  = cuda_str
print(f'PyTorch {pt_version}, CUDA {cuda_str}')

# torch-scatter / torch-sparse need precompiled binaries matching the runtime
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html -q
!pip install decord pytorchvideo fvcore iopath torch-geometric -q
!pip install -r {REPO_DIR}/requirements.txt -q
print('Dependencies installed.')

In [ ]:
# ── 5. Copy step embeddings to local storage (faster than reading from Drive) ──
import os
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)
!cp -r {STEP_EMBEDDINGS_DIR}/* {LOCAL_EMBEDDINGS_DIR}/
print(f"Copied: {len(os.listdir(LOCAL_EMBEDDINGS_DIR))} files → {LOCAL_EMBEDDINGS_DIR}")

In [ ]:
# ── 6. Verify pre-fusion cache (built by b3_hungarian_matching.ipynb) ──────
# The cache was built with sim_threshold=SIM_THRESHOLD by b3_hungarian_matching.
# Do NOT clear it — clearing would force a rebuild with random projector weights.
# To rebuild with a different threshold, re-run b3_hungarian_matching.ipynb.
import os, pathlib
cache_pt = list(pathlib.Path(CACHE_DIR).glob("*.pt")) if os.path.exists(CACHE_DIR) else []
if cache_pt:
    print(f"Cache ready: {len(cache_pt)} files in {CACHE_DIR}")
else:
    print(f"WARNING: cache not found at {CACHE_DIR}")
    print("Run b3_hungarian_matching.ipynb first.")

In [ ]:
# EgoVLP's FrozenInTime constructor loads a local ViT-B/16 checkpoint from
# /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth. Download it.
!mkdir -p {EGOVLP_REPO}/pretrained
!wget -q -nc -O {EGOVLP_REPO}/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -la {EGOVLP_REPO}/pretrained/

In [ ]:
# ── 7. WandB login ────────────────────────────────────────────────────────────
!wandb login

## Smoke test — 3 epochs (sanity check)

Verifica che:
- La pre-fusion cache sia completa (384 file `.pt`)
- Il `DAGClassifier` (DAGNN) giri senza errori con lo split train/val/test
- Loss e val AUC abbiano valori sensati dopo 3 epoch

**Tempo stimato:** ~1-2 min.

In [ ]:
SMOKE_OUTPUT_DIR = f'{DRIVE_ROOT}/step4/results_smoke_thr030'

In [ ]:
'''
%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$GRAPHS_DIR" "$EGOVLP_REPO" "$EGOVLP_CKPT" "$CACHE_DIR" "$SMOKE_OUTPUT_DIR"

cd /content/code

python -m extension.step4.train_dag_classifier \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --graphs_dir          "$4" \
    --egovlp_repo         "$5" \
    --egovlp_ckpt         "$6" \
    --cache_dir           "$7" \
    --output_dir          "$8" \
    --num_epochs   3  \
    --hidden_dim   128 \
    --num_layers   2  \
    --dropout      0.5 \
    --lr           1e-3 \
    --weight_decay 1e-4 \
    --batch_size   4  \
    --threshold    0.5 \
    --n_val        4  \
    --n_test       4  \
    --seed         42  \
    --patience     5  \
    --num_workers  2 \
    --enable_wandb
'''

## Full training — 50 epochs

Allena il `DAGClassifier` (DAGNN) con split train/val/test fisso (seed=42):
- **Train**: 16 ricette | **Val**: 4 ricette | **Test**: 4 ricette

Early stopping su val AUC (`--patience 15`).
Il modello migliore (best val AUC) viene salvato in `STEP4_OUTPUT_DIR/checkpoints/best.pt`.
Valutazione finale sul test set dopo il training — una volta sola.
Metriche e split salvati in `STEP4_OUTPUT_DIR/`.

**Tempo stimato:** ~15-30 min su T4.

In [ ]:
%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$GRAPHS_DIR" "$EGOVLP_REPO" "$EGOVLP_CKPT" "$CACHE_DIR" "$STEP4_OUTPUT_DIR"
cd /content/code

python -m extension.step4.train_dag_classifier \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --graphs_dir          "$4" \
    --egovlp_repo         "$5" \
    --egovlp_ckpt         "$6" \
    --cache_dir           "$7" \
    --output_dir          "$8" \
    --num_epochs   50  \
    --hidden_dim   128 \
    --num_layers   2   \
    --dropout      0.5 \
    --lr           1e-3 \
    --weight_decay 1e-4 \
    --batch_size   4   \
    --threshold    0.5 \
    --n_val        4   \
    --n_test       4   \
    --seed         42  \
    --patience     15  \
    --num_workers  2   \
    --enable_wandb

## Risultati — verifica e summary

In [ ]:
# ── Verifica checkpoint e split ───────────────────────────────────────────────
import pathlib, torch, json

ckpt_path = pathlib.Path(STEP4_OUTPUT_DIR) / 'checkpoints' / 'best.pt'
if ckpt_path.exists():
    saved = torch.load(ckpt_path, weights_only=False)
    print(f"Checkpoint: {ckpt_path.name}")
    print(f"  Best epoch  : {saved['epoch']}")
    print(f"  Val AUC     : {saved['val_metrics']['auc']:.4f}")
    print(f"  Val F1      : {saved['val_metrics']['f1']:.4f}")
    print(f"  Val Accuracy: {saved['val_metrics']['accuracy']:.4f}")
else:
    print(f"WARNING: checkpoint not found at {ckpt_path}")

split_path = pathlib.Path(STEP4_OUTPUT_DIR) / 'split_info.json'
if split_path.exists():
    info = json.load(open(split_path))
    print(f"\nSplit (seed={info['seed']})")
    print(f"  train : {info['train']}")
    print(f"  val   : {info['val']}")
    print(f"  test  : {info['test']}")

In [ ]:
# ── Leggi e stampa results.csv ────────────────────────────────────────────────
import pandas as pd

csv_path = f'{STEP4_OUTPUT_DIR}/results.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))